In [2]:
import duckdb
import polars as pl
import pandas as pd
import json
import os
import requests
curseforge_api_key = '$2a$10$Ut1BlDgVKpeGVSFKYfepDezg77aANwjF7WQwpQA75THOS0/4zTbHi'

### EXPLORING CURSEFORGE API

In [18]:
curseforge_url = 'https://api.curseforge.com'
headers = {'x-api-key': curseforge_api_key,
           'User-Agent': "liamxaro/Minecraft-data-platform/(contact: stormcraftmods@gmail.com)"}
minecraft_id = 432
minecraft_slug = 'minecraft'
params = {'index': 0,
          'pageSize': 50}



In [19]:
r1= requests.get(f"{curseforge_url}/v1/games/{minecraft_id}",
                 headers=headers,
                 params=params,
                 timeout=30)

r1.raise_for_status()
payload = r1.json()


In [20]:
payload

{'data': {'id': 432,
  'name': 'Minecraft',
  'slug': 'minecraft',
  'dateModified': '2025-05-04T08:40:05.227Z',
  'assets': {'iconUrl': 'https://media.forgecdn.net/avatars/1070/13/638605220544214742.webp',
   'tileUrl': 'https://media.forgecdn.net/game-box-art/432_bc6c42ce-5dd9-496a-a5cd-a51cf3584008.jpg',
   'coverUrl': 'https://media.forgecdn.net/game-covers/432_5076448d-feda-439b-9d63-ce86b70459ef.webp'},
  'status': 6,
  'apiStatus': 2,
  'supportedFeatures': {'supportModSubscriptions': False}}}

In [21]:
response = requests.get(
    f"{curseforge_url}/v1/categories",
    headers=headers,
    params={
        "gameId": 432,
        "classesOnly": True,
    },
    timeout=30,
)

response.raise_for_status()

classes = response.json()["data"]

for project_class in classes:
    print(
        project_class["id"],
        project_class["name"],
        project_class["slug"],
    )

5 Bukkit Plugins bukkit-plugins
17 Worlds worlds
12 Resource Packs texture-packs
4546 Customization customization
6945 Data Packs data-packs
4559 Addons mc-addons
4471 Modpacks modpacks
6552 Shaders shaders
6 Mods mc-mods


In [22]:
response = requests.get(
    f"{curseforge_url}/v1/mods/search",
    headers=headers,
    params={
        "gameId": 432,
        "classId": 6,
        "index": 0,
        "pageSize": 50,
    },
    timeout=30,
)

response.raise_for_status()

payload = response.json()


In [23]:
payload

{'data': [{'screenshots': [],
   'id': 1546263,
   'gameId': 432,
   'name': '"Arizona" Music Disc',
   'slug': 'arizona-music-disc',
   'links': {'websiteUrl': 'https://www.curseforge.com/minecraft/mc-mods/arizona-music-disc',
    'wikiUrl': None,
    'issuesUrl': None,
    'sourceUrl': None},
   'summary': 'A simple music mod, adding the song "Arizona" by Arcane Toaster. Used with permission!',
   'status': 4,
   'downloadCount': 446,
   'isFeatured': False,
   'primaryCategoryId': 424,
   'categories': [{'id': 424,
     'gameId': 432,
     'name': 'Cosmetic',
     'slug': 'cosmetic',
     'url': 'https://www.curseforge.com/minecraft/mc-mods/cosmetic',
     'iconUrl': 'https://media.forgecdn.net/avatars/6/39/635351497555976928.png',
     'dateModified': '2014-05-08T17:42:35.597Z',
     'isClass': False,
     'classId': 6,
     'parentCategoryId': 6}],
   'classId': 6,
   'authors': [{'id': 139715122,
     'name': 'birdmuncher',
     'url': 'https://www.curseforge.com/members/birdmunc

In [24]:
import requests


minecraft_game_id = 432
mods_class_id = 6


# Get every category underneath the Minecraft Mods class
categories_response = requests.get(
    f"{curseforge_url}/v1/categories",
    headers=headers,
    params={
        "gameId": minecraft_game_id,
        "classId": mods_class_id,
    },
    timeout=30,
)

categories_response.raise_for_status()

categories = categories_response.json()["data"]


category_counts = []


for category in categories:
    category_id = category["id"]
    category_name = category["name"]
    parent_category_id = category["parentCategoryId"]

    response = requests.get(
        f"{curseforge_url}/v1/mods/search",
        headers=headers,
        params={
            "gameId": minecraft_game_id,
            "classId": mods_class_id,
            "categoryId": category_id,
            "index": 0,
            "pageSize": 1,
        },
        timeout=30,
    )

    response.raise_for_status()

    payload = response.json()
    total_count = payload["pagination"]["totalCount"]

    category_counts.append(
        {
            "category_id": category_id,
            "category_name": category_name,
            "parent_category_id": parent_category_id,
            "total_count": total_count,
            "is_capped": total_count >= 10_000,
        }
    )


category_counts = sorted(
    category_counts,
    key=lambda row: row["total_count"],
    reverse=True,
)


for category in category_counts:
    capped_label = " <-- CAPPED" if category["is_capped"] else ""

    print(
        f"{category['category_id']:>6} | "
        f"{category['category_name']:<35} | "
        f"{category['total_count']:>7,}"
        f"{capped_label}"
    )

   434 | Armor, Tools, and Weapons           |  10,000 <-- CAPPED
   426 | Addons                              |  10,000 <-- CAPPED
   411 | Mobs                                |  10,000 <-- CAPPED
  5191 | Utility & QoL                       |  10,000 <-- CAPPED
  4906 | MCreator                            |  10,000 <-- CAPPED
   425 | Miscellaneous                       |  10,000 <-- CAPPED
   424 | Cosmetic                            |  10,000 <-- CAPPED
   408 | Ores and Resources                  |  10,000 <-- CAPPED
   422 | Adventure and RPG                   |  10,000 <-- CAPPED
   419 | Magic                               |  10,000 <-- CAPPED
   435 | Server Utility                      |   9,990
   436 | Food                                |   9,170
   406 | World Gen                           |   8,704
   412 | Technology                          |   8,647
   409 | Structures                          |   6,153
   407 | Biomes                              |   5,393
   421 | A